# OptiCrop: Smart Agricultural Production Optimization Engine

## Epic 1: Define Problem and Understanding

### Problem Statement
Farmers struggle to determine the most suitable crop based on soil nutrients and climate conditions, leading to financial losses and low agricultural yield. This project aims to develop an AI-driven crop recommendation system.

### Key Environmental Parameters
- **Nitrogen (N)**: Ratio of Nitrogen content in soil
- **Phosphorous (P)**: Ratio of Phosphorous content in soil  
- **Potassium (K)**: Ratio of Potassium content in soil
- **Temperature**: Temperature in degree Celsius
- **Humidity**: Relative humidity in %
- **pH**: pH value of the soil
- **Rainfall**: Rainfall in mm

### Expected Outcomes
1. Recommend the most suitable crop for given soil and environmental conditions
2. Maximize yields and resource efficiency
3. Support sustainable agricultural practices
4. Provide intelligent data-driven recommendations for farmers

## Epic 2: Data Collection and Analysis

### Story 1: Download the Dataset
The Crop_recommendation.csv dataset is used from Kaggle, containing 2200 records with 7 features and 22 crop labels.

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

In [ ]:
# Story 3: Read the Dataset
df = pd.read_csv('../dataset/Crop_recommendation.csv')
print(f'Dataset Shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Dataset information
print('Dataset Info:')
print(df.info())
print(f'\nColumn names: {list(df.columns)}')
print(f'\nUnique crops: {df["label"].nunique()}')
print(f'Crop list: {sorted(df["label"].unique())}')

In [ ]:
# Descriptive statistics
df.describe()

### Story 4: Univariate Analysis

In [ ]:
# Distribution of numerical features
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

for i, feature in enumerate(features):
    row, col = i // 4, i % 4
    sns.histplot(df[feature], kde=True, ax=axes[row, col], color='steelblue')
    axes[row, col].set_title(f'Distribution of {feature}')

axes[1, 3].axis('off')
plt.tight_layout()
plt.savefig('../app/static/univariate_dist.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Count of each crop
plt.figure(figsize=(14, 6))
sns.countplot(data=df, x='label', palette='viridis')
plt.xticks(rotation=90)
plt.title('Distribution of Crops')
plt.xlabel('Crop')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../app/static/crop_count.png', dpi=150, bbox_inches='tight')
plt.show()

### Story 5: Bivariate Analysis

In [ ]:
# Relationship between humidity and crop labels
plt.figure(figsize=(16, 6))
sns.scatterplot(data=df, x='humidity', y='rainfall', hue='label', palette='tab20', s=50)
plt.title('Humidity vs Rainfall by Crop')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../app/static/bivariate_humidity_rainfall.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Temperature vs Rainfall by crop
plt.figure(figsize=(16, 6))
sns.scatterplot(data=df, x='temperature', y='N', hue='label', palette='tab20', s=50)
plt.title('Temperature vs Nitrogen by Crop')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Story 6: Multivariate Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../app/static/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Pairplot of key features
sns.pairplot(df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']], diag_kind='kde')
plt.suptitle('Pairplot of Agricultural Features', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for each feature
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for i, feature in enumerate(features):
    row, col = i // 4, i % 4
    sns.boxplot(data=df, y=feature, ax=axes[row, col], color='lightcoral')
    axes[row, col].set_title(f'Boxplot of {feature}')

axes[1, 3].axis('off')
plt.tight_layout()
plt.savefig('../app/static/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## Epic 3: Data Pre-processing

### Story 1: Check for Null Values

In [ ]:
# Check for null values
print('Dataset Shape:', df.shape)
print('\nNull Values:')
print(df.isnull().sum())
print('\nData Types:')
print(df.dtypes)

### Story 2: Detect and Treat Outliers

In [ ]:
# Detect outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return len(outliers), lower_bound, upper_bound

print('Outlier Detection using IQR method:')
print('=' * 50)
for feature in features:
    count, lb, ub = detect_outliers_iqr(df, feature)
    print(f'{feature}: {count} outliers (bounds: [{lb:.2f}, {ub:.2f}])')

In [ ]:
# Apply log transformation to Potassium to handle outliers
df['K_log'] = np.log1p(df['K'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['K'], kde=True, ax=axes[0], color='coral')
axes[0].set_title('Original Potassium Distribution')
sns.histplot(df['K_log'], kde=True, ax=axes[1], color='steelblue')
axes[1].set_title('Log-Transformed Potassium Distribution')
plt.tight_layout()
plt.show()

### Story 3: Extract Seasonal Crop Information

In [ ]:
# Extract seasonal information based on temperature and rainfall
def assign_season(temp, rainfall):
    if temp > 25 and rainfall > 100:
        return 'rainy'
    elif temp > 25:
        return 'summer'
    elif temp < 20:
        return 'winter'
    else:
        return 'autumn'

df['season'] = df.apply(lambda row: assign_season(row['temperature'], row['rainfall']), axis=1)

# Analyze crops by season
season_crop = df.groupby(['season', 'label']).size().reset_index(name='count')
top_season_crops = season_crop.loc[season_crop.groupby('season')['count'].idxmax()]
print('Top crops per season:')
print(top_season_crops.to_string(index=False))

In [ ]:
# Season distribution
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='season', palette='Set2')
plt.title('Distribution of Seasons in Dataset')
plt.show()

### Story 4: Split Dataset into Training and Testing Sets

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode the target variable
le = LabelEncoder()
df['crop_encoded'] = le.fit_transform(df['label'])

# Features and target
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['crop_encoded']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f'Training set size: {X_train.shape[0]}')
print(f'Testing set size: {X_test.shape[0]}')
print(f'Number of features: {X_train.shape[1]}')
print(f'Number of classes: {len(le.classes_)}')

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Features scaled successfully!')

## Epic 4: Model Building

### Story 1: K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans

# Elbow Method to determine optimal number of clusters
wcss = []
k_range = range(1, 22)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_train_scaled)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, wcss, 'bo-', linewidth=2, markersize=6)
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.title('Elbow Method for Optimal k')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Apply K-Means with optimal k (22 crop types)
optimal_k = 22
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X)

print(f'K-Means clustering applied with k={optimal_k}')
print(f'\nCluster distribution:')
print(df['cluster'].value_counts().sort_index())

### Story 2: Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Train Logistic Regression model
lr_model = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluation
print('Logistic Regression Results:')
print('=' * 50)
print(f'Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'Precision (weighted): {precision_score(y_test, y_pred_lr, average="weighted"):.4f}')
print(f'Recall (weighted): {recall_score(y_test, y_pred_lr, average="weighted"):.4f}')
print(f'F1-Score (weighted): {f1_score(y_test, y_pred_lr, average="weighted"):.4f}')

### Story 3: Additional Models for Comparison

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Train multiple models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, average='weighted'),
        'Recall': recall_score(y_test, y_pred, average='weighted'),
        'F1-Score': f1_score(y_test, y_pred, average='weighted')
    }
    
    print(f'\n{name}:')
    print(f'  Accuracy:  {results[name]["Accuracy"]:.4f}')
    print(f'  Precision: {results[name]["Precision"]:.4f}')
    print(f'  Recall:    {results[name]["Recall"]:.4f}')
    print(f'  F1-Score:  {results[name]["F1-Score"]:.4f}')

In [ ]:
# Compare models visually
results_df = pd.DataFrame(results).T
results_df.plot(kind='bar', figsize=(12, 6), colormap='viridis')
plt.title('Model Comparison - Classification Metrics')
plt.ylabel('Score')
plt.ylim(0.8, 1.05)
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('../app/static/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nModel Comparison Summary:')
print(results_df.to_string())

In [ ]:
# Confusion Matrix for best model
best_model_name = results_df['Accuracy'].idxmax()
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print(f'\nBest Model: {best_model_name} (Accuracy: {results[best_model_name]["Accuracy"]:.4f})')

In [ ]:
# Classification report for best model
print(f'\nClassification Report - {best_model_name}:')
print('=' * 60)
print(classification_report(y_test, y_pred_best, target_names=le.classes_))

### Story 4: Save the Best Model

In [ ]:
import pickle

# Save the best model and label encoder
model_data = {
    'model': best_model,
    'label_encoder': le,
    'scaler': scaler,
    'model_name': best_model_name
}

with open('../app/model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print(f'Best model ({best_model_name}) saved to app/model.pkl')

In [ ]:
# Verify saved model
with open('../app/model.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

loaded_model = loaded_data['model']
loaded_le = loaded_data['label_encoder']
loaded_scaler = loaded_data['scaler']

# Test prediction
test_sample = X_test.iloc[0:1]
test_scaled = loaded_scaler.transform(test_sample)
prediction = loaded_model.predict(test_scaled)
predicted_crop = loaded_le.inverse_transform(prediction)

print(f'Test Input: {test_sample.values[0]}')
print(f'Predicted Crop: {predicted_crop[0]}')
print(f'Actual Crop: {le.inverse_transform([y_test.iloc[0]])[0]}')
print('\nModel verification complete!')

## Summary

- **Dataset**: 2200 records, 7 features, 22 crop types
- **Preprocessing**: Null check (none found), outlier detection (Potassium), log transform, seasonal extraction, train/test split (80/20)
- **Models Trained**: Logistic Regression, Random Forest, KNN, Decision Tree, K-Means Clustering
- **Best Model**: Saved as `model.pkl` for Flask deployment
- **Next Step**: Flask web application deployment